### Define Schema for Our Sample Streaming Data

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_defined = StructType([
    StructField("File", StringType(), True),
    StructField("Shop", StringType(), True),
    StructField("Sale_count", IntegerType(), True)
])

### Creatting or removing  the Folder Structure in volume

In [0]:
# Remove existing folders
dbutils.fs.rm("/Volumes/tri/tri/tri_files/INPUT/stream_checkpoint/", True)
dbutils.fs.rm("/Volumes/tri/tri/tri_files/INPUT/stream_read/", True)
dbutils.fs.rm("/Volumes/tri/tri/tri_files/INPUT/stream_write/", True)

# Create folders again
dbutils.fs.mkdirs("/Volumes/tri/tri/tri_files/INPUT/stream_checkpoint/")
dbutils.fs.mkdirs("/Volumes/tri/tri/tri_files/INPUT/stream_read/")
dbutils.fs.mkdirs("/Volumes/tri/tri/tri_files/INPUT/stream_write/")

### Read Streaming Data

In [0]:
df = (
    spark.readStream
    .format("csv")
    .schema(schema_defined)
    .option("header", True)
    .option("sep", ";")
    .load("/Volumes/tri/tri/tri_files/INPUT/stream_read/")
)



###  writing streaming Data

In [0]:
df4 = (
    df.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "/Volumes/tri/tri/tri_files/INPUT/stream_write/")
    .option("checkpointLocation", "/Volumes/tri/tri/tri_files/INPUT/stream_checkpoint/")
    .trigger(availableNow=True)
    .start()
)

df4.awaitTermination()

### Verify the Written Stream Output Data

In [0]:
display(
    spark.read
    .format("parquet")
    .load("/Volumes/tri/tri/tri_files/INPUT/stream_write/")
)

**🧠 Checkpoint in Spark Streaming**

**What is a Checkpoint?**

- A checkpoint stores the progress and state of a streaming job.
- It helps Spark continue from where it stopped if the job fails or restarts.
- Think of it as a bookmark for the streaming job.
- Why is it needed?
- •	Remembers already processed data.
- •	Helps recover after job failure.
- •	Avoids unnecessarily processing the same data again during recovery.
- •	Maintains information required for stateful operations like aggregations.
- What does it store?
- •	Offsets / progress: how much source data is processed.
- •	Metadata: information about the streaming query.
- •	Commit information: successfully completed batches.
- •	State: information used by aggregations and other stateful operations.
- Where is it stored?
- Checkpoint should be stored in persistent storage, such as ADLS in Azure Databricks.
- Python
- .option("checkpointLocation", "/checkpoint/orders")
Target vs Checkpoint
Target	Checkpoint
Stores actual output data	Stores streaming progress/state
Contains business records	Contains recovery information
Important Interview Points
•	Each streaming query should normally have its own checkpoint location.
•	If a job fails, Spark uses the checkpoint to recover and continue.
•	Don't delete the checkpoint casually. The restarted stream can be treated as a new query.
•	Checkpoint is especially important for stateful operations.
⭐ Interview Answer
Checkpoint stores the progress and state of a streaming query. If the job fails or restarts, Spark uses the checkpoint to recover and continue from the previous progress.
Easy memory:
